In [1]:

import pandas as pd
import numpy as np 
import seaborn as sns
import joblib
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime, timedelta
import function

In [2]:
import os
folder_path = "..\\data"
valid_sample = []
start_time = 0
for file in os.listdir(folder_path):
    path = os.path.join(folder_path,file)
    sample = pd.read_csv(path)
    start = min(sample['start_ns'])
    sample['start_ns'] = sample['start_ns'].transform(lambda x: (x - start))
    if start_time < start:
        end = max(sample['start_ns'])
        start_time = end
    else: 
        sample['start_ns'] = sample['start_ns'].transform(lambda x: (x + start_time))
        end = max(sample['start_ns'])
        start_time = end

    valid_sample.append(sample)
df = pd.concat(valid_sample,axis=0)


In [3]:
df.dropna(axis=0,inplace=True)

In [4]:
def hex_to_int(hex_str):
    """Convert hex string to integer efficiently - handles multiple formats"""
    return hex_str.apply(lambda x: int(str(x).replace("0x",''),16)).astype(np.int64)

In [5]:
df['PFN_int'] = hex_to_int(df['PFN'])
# df['Va_L1'] = df['VA'].apply(lambda x: f'{(x >> 12)& 0x1FF}')
# df['Va_L2'] = df['VA'].apply(lambda x: f'{(x >> 21)& 0x1FF}') 
# df['Va_L3'] = df['VA'].apply(lambda x: f'{(x >> 30)& 0x1FF}') 
# df['Va_L4'] = df['VA'].apply(lambda x: f'{(x >> 39)& 0xFFFFFFFF}') 

# page frame number are descride to predict actaul pfn using classified into sub region due to avoid pfn randomness. 

df['PFN_Top_region']=df['PFN_int'].apply(lambda x:(x >> 20) & 0xFF)
df['PFN_slice_4']=df['PFN_int'].apply(lambda x:(x >> 16) & 0xF )
df['PFN_slice_3']=df['PFN_int'].apply(lambda x:(x >> 12) & 0xF )
df['PFN_slice_2']=df['PFN_int'].apply(lambda x:(x >> 8) & 0xF )
df['PFN_slice_1']=df['PFN_int'].apply(lambda x:(x >> 4) & 0xF )
df['PFN_slice_0']=df['PFN_int'].apply(lambda x: x  & 0xF )

(
    df['PFN_slice_0'].unique(),
    df['PFN_slice_1'].unique(),
    df['PFN_slice_2'].unique(),
    df['PFN_slice_3'].unique(),
    df['PFN_slice_4'].unique(),
    df['PFN_Top_region'].unique()
)

(array([ 1,  2,  5, 13,  7, 12, 11,  4,  8,  9,  0, 15, 14, 10,  6,  3],
       dtype=int64),
 array([15,  5, 13,  3,  6,  0, 11,  2, 14,  4,  1,  9,  8, 10,  7, 12],
       dtype=int64),
 array([ 2, 14,  7,  8,  1, 15, 10, 12, 13,  3, 11,  6,  0,  4,  9,  5],
       dtype=int64),
 array([14,  4,  9,  5,  3,  7,  2,  1, 13,  6, 10,  0, 15, 11, 12,  8],
       dtype=int64),
 array([ 9,  1,  5,  3, 13, 11,  0, 10,  4,  8, 12,  6,  2,  7],
       dtype=int64),
 array([0, 1], dtype=int64))

In [ ]:

# 1) Fix outlier latencies (convert to ms)
mask = df['latency_ns'] > 200*1e6   # > 200ms
df.loc[mask, 'latency_ns'] = df.loc[mask, 'latency_ns'] / 1e6   # ns → ms
df[['PID','COMM','VA','PFN','mapping','folio_index','start_ns','latency_ns']].to_csv('../_filter/swap_log_1.csv', index=False)

In [8]:

df = df.sort_values(by="start_ns", ascending=True).reset_index(drop=True)

In [9]:
fig5 = px.line(df, x="start_ns", y="PFN_int",
                  hover_data=["PID","COMM"],
                  title="PFN Distribution (Group by PID)",
                  # color=df['PID'].astype('str'),
                #   animation_frame=df['PFN_slice_4'].astype('str'),
                  template="plotly_dark")
fig5.update_layout(xaxis_title="start_sec (time)", yaxis_title="PFN")
fig5.show()

In [2]:


#----------------- csv load and null remove -----------
df = pd.read_csv('swap_log_1.csv')
df.head(5)
df.dropna(inplace=True)

#------------------ data formatting ---------------------

# Apply conversions more efficiently with error handling
hex_columns = ['PID', 'VA', 'PFN', 'mapping']
for col in hex_columns:
    df[col] = function.hex_to_int(df[col])

# Convert numeric columns with error handling
df['start_ns'] = pd.to_numeric(df['start_ns'], errors='coerce').fillna(0).astype(np.int64)
df['folio_index'] = pd.to_numeric(df['folio_index'], errors='coerce').fillna(0).astype(np.int16)
df['latency_ns'] = pd.to_numeric(df['latency_ns'], errors='coerce').fillna(0).astype(np.int64)

# Remove any rows that might have become NaN during conversion
df = df.dropna()

# Calculate derived columns
df['latency_ms'] = np.round(df['latency_ns'] / 1e6, 2)
df['latency_mcs'] = np.round(df['latency_ns'] / 1e3, 2)

# Time string conversion
df['time_str'] = pd.to_timedelta(df['start_ns'], unit='ns').dt.total_seconds().apply(
    lambda x: f"{int(x // 3600):02d}:{int((x % 3600) // 60):02d}:{int(x % 60):02d}"
)

print(f"DataFrame shape: {df.shape}")
print(f"Unique PIDs: {df['PID'].nunique()}")

#--------------- data transformation ---------------------

# Initialize scalers
scaler_va = MinMaxScaler()
scaler_pfn = MinMaxScaler()
scaler_mapping = MinMaxScaler()
scaler_index = MinMaxScaler()
scaler_latency = MinMaxScaler()

# Fit and transform data
df['va_transform'] = scaler_va.fit_transform(df[['VA']])
df['pfn_transform'] = scaler_pfn.fit_transform(df[['PFN']])
df['index_transform'] = scaler_index.fit_transform(df[['folio_index']])
df['mapping_transform'] = scaler_mapping.fit_transform(df[['mapping']])
df['latency_transform'] = scaler_latency.fit_transform(df[['latency_mcs']])

# Save scalers for inverse transformation
joblib.dump(scaler_va, 'pkl/scaler_va.pkl')
joblib.dump(scaler_pfn, 'pkl/scaler_pfn.pkl')
joblib.dump(scaler_index, 'pkl/scaler_index.pkl')
joblib.dump(scaler_mapping, 'pkl/scaler_mapping.pkl')
joblib.dump(scaler_latency, 'pkl/scaler_latency.pkl')


# ---------- Create sequences with X=10, y=5 ----------
SEQ_LEN_X = 40  # Input sequence length (past events)
SEQ_LEN_Y = 5   # Output sequence length (future events to predict)
X, y = [], []
pid_indices = []

print(f"\nCreating sequences with X_len={SEQ_LEN_X}, y_len={SEQ_LEN_Y}...")

for pid in df["PID"].unique():
    pid_data = df[df["PID"] == pid]
    pid_values = pid_data[["va_transform", "pfn_transform", "index_transform", "mapping_transform", "latency_transform"]].values
    n = len(pid_values)

    # Only create sequences if we have enough data
    if n >= (SEQ_LEN_X + SEQ_LEN_Y):
        # Iterate over each possible starting point
        for i in range(SEQ_LEN_X, n - SEQ_LEN_Y + 1):
            # Input sequence: past SEQ_LEN_X events
            input_seq = pid_values[i-SEQ_LEN_X:i]
            # Target sequence: next SEQ_LEN_Y events
            target_seq = pid_values[i:i+SEQ_LEN_Y]
            
            X.append(input_seq)
            y.append(target_seq)
            pid_indices.append((pid, i-SEQ_LEN_X, i))

X = np.array(X)
y = np.array(y)

print(f"Created {len(X)} sequences")


# ---------- Example usage of inverse transformation ----------
if len(X) > 0:
    print(f"\n=== SEQUENCE INFORMATION ===")
    print(f"X shape: {X.shape}")  # Should be (samples, 10, 5)
    print(f"y shape: {y.shape}")  # Should be (samples, 5, 5)
    
    # Display first sequence in table format
    function.display_sequence_table(X[0], "INPUT", 0)
    function.display_sequence_table(y[0], "TARGET", 0)
    
    # Show sequence relationship
    print(f"\n=== SEQUENCE RELATIONSHIP ===")
    sample_pid, start_idx, target_start_idx = pid_indices[0]
    print(f"PID: {sample_pid}")
    print(f"Input X[0]: Events {start_idx} to {start_idx + SEQ_LEN_X - 1} (t-{SEQ_LEN_X} to t-1)")
    print(f"Target y[0]: Events {target_start_idx} to {target_start_idx + SEQ_LEN_Y - 1} (t to t+{SEQ_LEN_Y-1})")

# # Save the sequences for later use
np.save('npy/X_sequences.npy', X)
np.save('npy/y_sequences.npy', y)
joblib.dump(pid_indices, 'pkl/pid_indices.pkl')
joblib.dump({'SEQ_LEN_X': SEQ_LEN_X, 'SEQ_LEN_Y': SEQ_LEN_Y}, 'pkl/sequence_config.pkl')


# print(f"\nData preprocessing completed successfully!")
# print(f"Total sequences: {len(X)}")
# print(f"Input shape: {X.shape}")
# print(f"Target shape: {y.shape}")



DataFrame shape: (3662, 11)
Unique PIDs: 40

Creating sequences with X_len=40, y_len=5...
Created 2540 sequences

=== SEQUENCE INFORMATION ===
X shape: (2540, 40, 5)
y shape: (2540, 5, 5)

INPUT Sequence [0] - 40 Events
 Time_Step         VA_Hex PFN_Hex  Folio_Index Mapping_Hex  Latency_ns
         0 0x6e7bfba4a996 0x9e522            0  0x203008ae   804950000
         1 0x6e7bfba47c98  0x3762            0  0xb80da4f5   851716000
         2 0x6e7bfba46d98 0x15271            0  0xb80da4f5   850611000
         3 0x6e7bfba44f99 0xb43c7            0  0xb80da4f5   848544000
         4 0x6e7bfba4409a 0x35bd8            0  0xb80da4f5   847216000
         5 0x6e7bfba4319a 0x9e52e            0  0xb80da4f5   845056000
         6 0x6e7bfba4229b 0x57bf3            0  0xb80da4f5   842845000
         7 0x6e7bfba48b97 0xd2289            0  0xb80da4f5   853001000
         8 0x6e7bfba50393 0x99446            0  0x203008ae   826553000
         9 0x6e7bfba4f493 0xd3469            0  0x203008ae   824486000

['pkl/sequence_config.pkl']